# 02 · CRUD en MongoDB Atlas

Colección `events` (documentos originales). Driver: `pymongo`.

**Justificación:** el payload varía por tipo de evento → esquema flexible, consultas por campos anidados, agregaciones.

In [ ]:
import sys, os
sys.path.insert(0, "../src")          # módulos del proyecto (config, extract, clean, ...)
import config
from datetime import datetime, timezone, date
from pprint import pprint

In [ ]:
db = config.mongo_db()
col = db.events
print('BD:', db.name, '| documentos:', col.estimated_document_count())

## C · Crear documento

In [ ]:
demo_id = 'demo-' + datetime.now().strftime('%H%M%S')
doc = {'_id': demo_id, 'type': 'WatchEvent', 'actor': {'id': 1, 'login': 'estudiante_utec'},
       'repo': {'id': 1, 'name': 'utec/bigdata-demo'}, 'public': True,
       'created_at': datetime.now(timezone.utc), 'payload': {'action': 'started'}}
res = col.insert_one(doc)
print('insertado:', res.inserted_id)

## R · Consultar documentos

In [ ]:
pprint(col.find_one({'_id': demo_id}))

In [ ]:
# Consulta con filtro + proyección + orden
for d in col.find({'type': 'PushEvent'}, {'actor.login': 1, 'repo.name': 1, 'created_at': 1}).sort('created_at', -1).limit(5):
    print(d)

In [ ]:
# Agregación: eventos por tipo
pipeline = [{'$group': {'_id': '$type', 'n': {'$sum': 1}}}, {'$sort': {'n': -1}}, {'$limit': 8}]
pprint(list(col.aggregate(pipeline)))

## U · Actualizar documento

In [ ]:
res = col.update_one({'_id': demo_id}, {'$set': {'payload.action': 'stopped', 'revisado': True}, '$inc': {'veces_editado': 1}})
print('matched:', res.matched_count, '| modified:', res.modified_count)
pprint(col.find_one({'_id': demo_id}))

## D · Eliminar documento

In [ ]:
res = col.delete_one({'_id': demo_id})
print('eliminados:', res.deleted_count)
print('existe?', col.find_one({'_id': demo_id}) is not None)